In [ ]:
from flask import Flask, request, jsonify
from opentrons import simulate

app = Flask(__name__)

@app.route('/run_protocol', methods=['POST'])
def run_protocol():
    data = request.json
    well = data.get("well", "A1")
    volume = data.get("volume", 50)

    protocol_context = simulate.get_protocol_api("2.13")

    tip_rack = protocol_context.load_labware("opentrons_96_tiprack_300ul", "1")
    pipette = protocol_context.load_instrument("p300_single", "right", tip_racks=[tip_rack])
    plate = protocol_context.load_labware("corning_96_wellplate_360ul_flat", "2")

    pipette.pick_up_tip()
    pipette.aspirate(volume, plate["A1"])
    pipette.dispense(volume, plate[well])
    pipette.drop_tip()

    return jsonify({"status": "Protocol executed", "well": well, "volume": volume})

if __name__ == '__main__':
    app.run(host="0.0.0.0", port=5000, debug=True)
